In [1]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from scipy import stats as st

In [2]:
df_telecom_clients = pd.read_csv('../data/telecom_clients.csv')
df_telecom_dataset_new = pd.read_csv('../data/telecom_dataset_new.csv')

In [3]:
df_telecom_clients.head()

,user_id,tariff_plan,date_start
0,166713,A,2019-08-15
1,166901,A,2019-08-23
2,168527,A,2019-10-29
3,167097,A,2019-09-01
4,168193,A,2019-10-16


In [4]:
df_telecom_dataset_new.head()

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.0,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.0,True,3,0,25


In [5]:
df_telecom_clients.info()

<class 'pandas.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      732 non-null    int64
 1   tariff_plan  732 non-null    str  
 2   date_start   732 non-null    str  
dtypes: int64(1), str(2)
memory usage: 17.3 KB


In [6]:
df_telecom_dataset_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  str    
 2   direction            53902 non-null  str    
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(1), str(2)
memory usage: 3.3+ MB


In [7]:
df_telecom_clients['date_start'] = pd.to_datetime(df_telecom_clients['date_start'])
df_telecom_dataset_new['date'] = pd.to_datetime(df_telecom_dataset_new['date'])

In [8]:
df_telecom_clients.info()
df_telecom_dataset_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      732 non-null    int64         
 1   tariff_plan  732 non-null    str           
 2   date_start   732 non-null    datetime64[us]
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 17.3 KB
<class 'pandas.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype                    
---  ------               --------------  -----                    
 0   user_id              53902 non-null  int64                    
 1   date                 53902 non-null  datetime64[us, UTC+03:00]
 2   direction            53902 non-null  str                      
 3   internal             53785 non-null  object                   
 4   operator_id          45730 non-null  float64                  
 5   is_missed_call    

In [49]:
df_telecom_dataset_new['internal'].value_counts(dropna=False)

internal
False    47621
True      6164
NaN        117
Name: count, dtype: int64

In [51]:
df_telecom_dataset_new['internal'] = (
    df_telecom_dataset_new['internal'].astype('boolean')
)
df_telecom_dataset_new['internal'].dtype

BooleanDtype

In [52]:
df_telecom_dataset_new['internal'].value_counts(dropna=False)

internal
False    47621
True      6164
<NA>       117
Name: count, dtype: Int64

In [53]:
df_telecom_dataset_new['operator_id'].dropna().mod(1).value_counts()

operator_id
0.0    45730
Name: count, dtype: int64

In [58]:
df_telecom_dataset_new['operator_id'] = (
    df_telecom_dataset_new['operator_id'].astype('Int64')
)
df_telecom_dataset_new['operator_id'].dtype
df_telecom_dataset_new['operator_id'].isna().sum()

np.int64(8172)

### comentar algo sobre a conversao do tipo de dado aqui

In [59]:
df_telecom_dataset_new[
    ['calls_count', 'call_duration', 'total_call_duration']
].describe()

,calls_count,call_duration,total_call_duration
count,53902.000000,53902.000000,53902.000000
mean,16.451245,866.684427,1157.133297
std,62.917170,3731.791202,4403.468763
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,47.000000
50%,4.000000,38.000000,210.000000
75%,12.000000,572.000000,902.000000
max,4817.000000,144395.000000,166155.000000


# parei aqui 

### Investigate the null values

In [44]:
df_telecom_dataset_new.groupby(
    ['direction', 'is_missed_call'],
    dropna=False
).agg(
    register_total = ('user_id', 'size'),
    operator_missing = ('operator_id', lambda s: s.isna().sum())
)

register_total  operator_missing
direction is_missed_call                                  
in        False                    13342                73
          True                      8643              7899
out       False                    16992                49
          True                     14925               151

In [47]:
df_telecom_dataset_new.groupby('direction').agg(
    register_total = ('user_id', 'size'),
    operator_missing = ('operator_id', 'count')
).assign(
    operator_absent = lambda df:
        df['register_total'] - df['operator_missing']
)

,register_total,operator_missing,operator_absent
direction,,,
in,21985,14013,7972
out,31917,31717,200


In [43]:
missing_values = pd.DataFrame(
    {
    'missing_count': df_telecom_dataset_new.isna().sum(),
    'missing_percentage': (
        df_telecom_dataset_new.isna().mean() * 100
    ).round(2)
})
missing_values.sort_values(by='missing_count', ascending=False)

,missing_count,missing_percentage
operator_id,8172,15.16
internal,117,0.22
user_id,0,0.00
date,0,0.00
direction,0,0.00
is_missed_call,0,0.00
calls_count,0,0.00
call_duration,0,0.00
total_call_duration,0,0.00


In [48]:
df_telecom_dataset_new[
    df_telecom_dataset_new['internal'].isna()
].groupby(
    ['direction', 'is_missed_call'],
    dropna=False
).agg(
    register_total = ('user_id', 'size'),
    calls_total = ('calls_count', 'sum')
)

register_total  calls_total
direction is_missed_call                             
in        False                       59           64
          True                        56          164
out       True                         2            6

### Analise de valores ausentes
- A analise identificou 8.172 valores ausentes na coluna `operator_id`, correspondentes a 15,16% dos registros, e 117 valores ausentes na coluna internal, equivalentes a 0,22% da base original.
- A ausencia de `operator_id` esta concentrada nas chamadas recebidas, principalmente naquelas classificadas como perdidas. Como nao ha informacoes suficientes esses valores nao serao preenchidos. Os registros serao preservados na analise geral e nao serao utilizados nos indicadores que dependem da identificacao individual do operador. 
- O valores ausentes em internal representam 234 chamadas, de acordo com a soma da coluna `calls_count`. 
- A analise por categoria mostrou que os valores ausentes estao presentes em chamadas recebidas tanto perdidas quanto nao perdidas, e em dois registros de chamadas efetuadas perdidas. Nao foram encontrados valores ausentes nessa coluna entre as chamadas efetuadas nao perdidas. 

### Investigate the duplicate

In [17]:
duplicate_count = df_telecom_clients.duplicated().sum()

duplicate_percentage = duplicate_count / len(df_telecom_clients) * 100

print("Duplicatas:", duplicate_count)
print("Percentual:", duplicate_percentage)

Duplicatas: 0
Percentual: 0.0


In [18]:
df_telecom_clients[df_telecom_clients.duplicated(keep=False)].head(10)

,user_id,tariff_plan,date_start


In [27]:
duplicate_clients = (
    df_telecom_clients['user_id']
    .duplicated()
    .sum()
)
print(f"Number of duplicate user_id in df_telecom_clients: {duplicate_clients}")
print(f"Number of duplicate rows considering user_id: "
      f"{df_telecom_clients.duplicated(subset=['user_id'], keep=False).sum()}")
print(f"Number of unique users in df_telecom_clients: {df_telecom_clients['user_id'].nunique()}")
print(f"Total number of lines in df_telecom_clients: {len(df_telecom_clients)}")

Number of duplicate user_id in df_telecom_clients: 0
Number of duplicate rows considering user_id: 0
Number of unique users in df_telecom_clients: 732
Total number of lines in df_telecom_clients: 732


In [19]:
duplicate_count_dataset = df_telecom_dataset_new.duplicated().sum()

duplicate_percentage_dataset = duplicate_count_dataset / len(df_telecom_dataset_new) * 100

print("Duplicatas:", duplicate_count_dataset)
print("Percentual:", duplicate_percentage_dataset)

Duplicatas: 4900
Percentual: 9.090571778412675


- 4,900 duplicate records were found, representing approximately 9.09% of the database.

In [20]:
df_telecom_dataset_new[df_telecom_dataset_new.duplicated(keep=False)].head(10)

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
6,166377,2019-08-05 00:00:00+03:00,out,False,880020.0,True,8,0,50
8,166377,2019-08-05 00:00:00+03:00,out,False,880020.0,True,8,0,50
27,166377,2019-08-12 00:00:00+03:00,in,False,NaN,True,2,0,34
28,166377,2019-08-12 00:00:00+03:00,in,False,NaN,True,2,0,34
38,166377,2019-08-14 00:00:00+03:00,in,False,NaN,True,1,0,3
43,166377,2019-08-14 00:00:00+03:00,out,False,880026.0,False,10,1567,1654
44,166377,2019-08-14 00:00:00+03:00,out,False,880026.0,False,10,1567,1654
45,166377,2019-08-14 00:00:00+03:00,in,False,NaN,True,1,0,3
46,166377,2019-08-15 00:00:00+03:00,out,False,880026.0,False,11,1413,1473
51,166377,2019-08-15 00:00:00+03:00,out,False,880026.0,False,11,1413,1473


In [37]:
df_without_duplicates = df_telecom_dataset_new.drop_duplicates()

print('Original records: ', len(df_telecom_dataset_new))

print('Records without duplicates: ', len(df_without_duplicates))

missed_calls_before = df_telecom_dataset_new.loc[
    df_telecom_dataset_new['is_missed_call'] == True,
        'calls_count'
    ].sum()



missed_calls_after = df_without_duplicates.loc[
    df_without_duplicates['is_missed_call'] == True,
        'calls_count'
    ].sum()

print("Missed calls before:", missed_calls_before)
print("Missed calls after:", missed_calls_after)
print("Difference:", missed_calls_before - missed_calls_after)

Original records:  53902
Records without duplicates:  49002
Missed calls before: 412305
Missed calls after: 375918
Difference: 36387


- O total de chamadas perdidas diminuiu 36.387, aproximadamente 8,83% do total original de chamadas perdidas, sendo diferencas entre os dois cenarios calculados. Sendo significativo para a nossa analise: a presenca dessas duplicatas altera materialmente uma metrica importante do projeto. 

In [38]:
duplicate_rows = df_telecom_dataset_new[
    df_telecom_dataset_new.duplicated(keep=False)
]

duplicate_rows.groupby('direction').agg(
    duplicate_records=('user_id', 'size'),
    missed_calls=('calls_count', 'sum'),
    operators=('operator_id', 'nunique')
)

missed_by_operator_before = (
    df_telecom_dataset_new[
        (df_telecom_dataset_new['direction'] == 'in') &
        (df_telecom_dataset_new['is_missed_call'] == True)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
)

missed_by_operator_after = (
    df_without_duplicates[
        (df_without_duplicates['direction'] == 'in') &
        (df_without_duplicates['is_missed_call'] == True)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
)

comparison = pd.concat([
    missed_by_operator_before.rename('before'),
    missed_by_operator_after.rename('after')
], axis=1).fillna(0)

comparison['difference'] = comparison['before'] - comparison['after']

comparison.sort_values(by='difference', ascending=False).head(10)

,before,after,difference
operator_id,,,
944216.0,32,24,8
944226.0,38,30,8
885876.0,24,20,4
937860.0,8,5,3
913942.0,55,52,3
944220.0,21,18,3
951508.0,24,21,3
940658.0,13,10,3
919206.0,12,9,3


In [41]:
comparison['percentage_difference'] = (
    comparison['difference']
    .div(comparison['before'].replace(0, float('nan')))
    .mul(100)
)

print('Largest absolute difference:', comparison['difference'].max())

print('\nAffected operators:')
print((comparison['difference'] > 0).sum())
print('\nLargest percentage difference:', comparison['percentage_difference'].max())

comparison['rank_before'] = (
    comparison['before']
    .rank(method='min', ascending=False)
)
comparison['rank_after'] = (
    comparison['after']
    .rank(method='min', ascending=False)
)
comparison['rank_change'] = (
comparison['rank_before'] - comparison['rank_after']
)

print('\nLargest rank change:', comparison['rank_change'].abs().max())

Largest absolute difference: 8

Affected operators:
46

Largest percentage difference: 50.0

Largest rank change: 36.0


In [42]:
comparison.sort_values(by='rank_change', key=abs, ascending=False)[
    [
    'before',
    'after',
    'difference',
    'rank_before',
    'rank_after',
    'rank_change'
    ]
].head(10)

,before,after,difference,rank_before,rank_after,rank_change
operator_id,,,,,,
920726.0,2,1,1,94.0,130.0,-36.0
953464.0,2,1,1,94.0,130.0,-36.0
919792.0,2,1,1,94.0,130.0,-36.0
919896.0,2,1,1,94.0,130.0,-36.0
906294.0,2,1,1,94.0,130.0,-36.0
905566.0,2,1,1,94.0,130.0,-36.0
937812.0,2,1,1,94.0,130.0,-36.0
934430.0,2,1,1,94.0,130.0,-36.0
911142.0,3,2,1,68.0,90.0,-22.0


### Tratamento de registros duplicados

- Durante a analise exploratoria, foram identificados 4.900 registros duplicados exatos, correspondentes a aproximadamente 9,09% dos registros originais. A remocao dessas duplicatas reduziu o volume de registros de 53.902 para 49.002 e o total de chamadas perdidas de 412.305 para 375.918, uma diferenca de 36.387 chamadas.
- A comparacao por operador mostrou que as maiores mudancas de posicao no ranking ocorreram entre os operadores com poucas chamadas perdidas. Alguns operadores passaram de duas para uma chamada perdida e mudaram 36 posicoes, demonstrando que o ranking e sensivel a pequenas diferencas absolutas.
- Portanto para a analise principal, serao utilizados os registros sem duplicatas exatas, mantendo-se a base original preservada. Essa decisao se basea na possibilidade de duplicacao indevida, mas reconhece que os dados disponiveis nao permitem confirmar individualmente a origem de cada registro duplicado. 